In [1]:
"""
DVT Detection — Kaggle Multi-GPU (2x T4) Final Version
======================================================
Optimized for Kaggle Notebooks with Dual T4 GPUs.

KEY CHANGES FROM CPU VERSION:
─────────────────────────────────────────────
 1. KAGGLE PATHS      Updated to /kaggle/input/hoseini-dvt-trainingdata/...
 2. MULTI-GPU (DP)    Model wrapped in nn.DataParallel to use both T4s
 3. AMP RESTORED      Added autocast() and GradScaler for fast Tensor Core math
 4. BATCH SIZE        Increased from 8 to 64 to saturate the 32GB total VRAM
 5. NUM_WORKERS       Increased to 2 for faster dataloader IO on Linux
 6. DP FIXES          State dict saving/loading and freezing modified to 
                      handle the 'module.' prefix introduced by DataParallel.
"""
!pip install -q pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg
import os, math, warnings, gc
from pathlib import Path

import torch
import cv2
import numpy as np
import pandas as pd
import pydicom
import matplotlib
matplotlib.use("Agg")           # Non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler  # AMP for T4 acceleration

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, roc_curve, auc,
)
from sklearn.preprocessing import StandardScaler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings("ignore")
print(f"PyTorch {torch.__version__} | GPUs detected: {torch.cuda.device_count()}")


# ══════════════════════════════════════════════════════════════
# CELL 2 — Config (Updated for Kaggle & Multi-GPU)
# ══════════════════════════════════════════════════════════════
class Config:
    # ── Paths (Based on provided Kaggle structure) ────────────
    DICOM_DIR  = "/kaggle/input/datasets/mahdihoseinikhah/hoseini-dvt-trainingdata/traindata/train"
    CSV_PATH   = "/kaggle/input/datasets/mahdihoseinikhah/hoseini-dvt-trainingdata/traindata/train_clean.csv"
    OUTPUT_DIR = "/kaggle/working/results_final"

    # ── Image ─────────────────────────────────────────────────
    IMG_SIZE   = 224              

    # ── Model ─────────────────────────────────────────────────
    MODEL_NAME = "swin_tiny_patch4_window7_224"   
    PRETRAINED = True

    # ── Training (Multi-GPU Tuned) ────────────────────────────
    BATCH_SIZE     = 64           # Increased to fill 2x16GB T4 VRAM
    NUM_EPOCHS     = 15           
    FREEZE_EPOCHS  = 1            
    WEIGHT_DECAY   = 1e-2
    GRAD_CLIP      = 1.0
    LR_MAX         = 3e-4         # Slightly higher LR for larger batch size

    # ── Class imbalance (auto-computed) ───────────────────────
    DVT_POS_WEIGHT  = 13.0       
    FOCAL_GAMMA     = 2.0
    FOCAL_ALPHA     = 0.75
    LABEL_SMOOTHING = 0.05

    # ── Metadata fusion ───────────────────────────────────────
    USE_METADATA = True

    # ── Split ─────────────────────────────────────────────────
    VAL_RATIO  = 0.15
    TEST_RATIO = 0.15
    SEED       = 42

    # ── System (Kaggle Linux Specific) ────────────────────────
    DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    NUM_WORKERS = 2               # Safe optimal for Kaggle CPU/IO

    # ── Clinical threshold ────────────────────────────────────
    DECISION_THRESHOLD = None



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 35.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.7 MB/s eta 0:00:00
PyTorch 2.10.0+cu128 | GPUs detected: 2


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mahdihoseinikhah/hoseini-dvt-trainingdata/traindata/train_clean.csv
/kaggle/input/datasets/mahdihoseinikhah/hoseini-dvt-trainingdata/traindata/train/5ea6ff775bdab5fbb7e524cbfce7f12f01176d9899e82885209a4d6988130d9e.dcm
/kaggle/input/datasets/mahdihoseinikhah/hoseini-dvt-trainingdata/traindata/train/6dda0755e4ba407b9b11f0ac7b112079c7ab129f243dcd91824a9c7977ab995e.dcm
/kaggle/input/datasets/mahdihoseinikhah/hoseini-dvt-trainingdata/traindata/train/c677eeed99d25255aff6b8597ed1d1cf46974b0583caabd24e7c1aefdad9db7d.dcm
/kaggle/input/datasets/mahdihoseinikhah/hoseini-dvt-trainingdata/traindata/train/3d5513e76e3df091752056feb26077148cf8fc40608987faa5ab3567f5f0563d.dcm
/kaggle/input/datasets/mahdihoseinikhah/hoseini-dvt-trainingdata/traindata/train/bc8af6f0c70eb1e962a8d593ef821d09d61280d82d64c60998aa152d8fb4fc28.dcm
/kaggle/input/datasets/mahdihoseinikhah/hoseini-dvt-trainingdata/traindata/train/1cd199747398d58a595539f831eea49ab0b65b18134a2c2b10a1ecef96610e6b.dcm
/kaggle/i

In [3]:
# ══════════════════════════════════════════════════════════════
# CELL 3 — DICOM loader
# ══════════════════════════════════════════════════════════════
def load_dicom_as_rgb(path: str, target_size: int = 224) -> np.ndarray:
    """
    Safely loads DICOM. Returns a black image if the file is 
    compressed/corrupt to prevent OpenCV from crashing.
    """
    try:
        dcm = pydicom.dcmread(path)
        # This line is where the 'empty' error usually starts
        arr = dcm.pixel_array.astype(float) 
    except Exception:
        # CRITICAL: Stop here and return a blank image if loading failed
        return np.zeros((target_size, target_size, 3), dtype=np.uint8)

    # 1. Handle MONOCHROME1
    if "MONOCHROME1" in str(getattr(dcm, "PhotometricInterpretation", "")):
        arr = arr.max() - arr

    # 2. Basic Normalization
    arr = ((arr - arr.min()) / (arr.max() - arr.min() + 1e-8) * 255).astype(np.uint8)

    # 3. Convert to 3-channel RGB (OpenCV needs this)
    if arr.ndim == 2:
        arr = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
    
    # 4. Final Safety Check: If for any reason arr is still empty, return black
    if arr is None or arr.size == 0:
        return np.zeros((target_size, target_size, 3), dtype=np.uint8)

    return cv2.resize(arr, (target_size, target_size))

# ══════════════════════════════════════════════════════════════
# CELL 4 — Master dataframe
# ══════════════════════════════════════════════════════════════
def build_master_dataframe(dicom_dir: str, csv_path: str) -> pd.DataFrame:
    print("\n[1] Building master dataframe (CSV ↔ DICOM join)")

    df = pd.read_csv(csv_path)
    df["File Name"] = df["File Name"].apply(lambda x: Path(x).name)

    disk_files = {p.name: str(p) for p in Path(dicom_dir).rglob("*.dcm")}
    print(f"  CSV rows       : {len(df):,}")
    print(f"  DICOM on disk  : {len(disk_files):,}")

    df["full_path"] = df["File Name"].map(disk_files)
    n_missing = df["full_path"].isna().sum()
    if n_missing:
        print(f"  [warn] {n_missing} CSV rows have no matching file — dropped")
    df = df.dropna(subset=["full_path"])

    assert "Thrombosis" in df.columns, "'Thrombosis' column not found in CSV"
    df["label"] = df["Thrombosis"].astype(int)

    n_normal = (df["label"] == 0).sum()
    n_dvt    = (df["label"] == 1).sum()
    ratio    = n_normal / max(n_dvt, 1)
    Config.DVT_POS_WEIGHT = float(ratio)

    print(f"  Normal: {n_normal:4d}  |  DVT: {n_dvt:4d}  |  Ratio={ratio:.1f}:1")
    return df.reset_index(drop=True)


# ══════════════════════════════════════════════════════════════
# CELL 5 — Image validation
# ══════════════════════════════════════════════════════════════
def validate_images(df: pd.DataFrame, n_check: int = 25):
    print(f"\n[IMG CHECK] Sampling {n_check} DICOMs...")
    sample = df.sample(min(n_check, len(df)), random_state=Config.SEED)
    black_count = 0
    for _, row in sample.iterrows():
        img = load_dicom_as_rgb(row["full_path"], 64)
        if img.mean() < 5.0:
            black_count += 1

    pct = 100 * black_count / len(sample)
    if pct > 40:
        raise RuntimeError(
            f"\n⚠ {pct:.0f}% of sampled images are BLACK.\n"
            "Ensure pylibjpeg is installed in Kaggle.\n"
        )
    print(f"  OK — {black_count}/{len(sample)} black. Images loading correctly.")


# ══════════════════════════════════════════════════════════════
# CELL 6 — IMAGE CACHE
# ══════════════════════════════════════════════════════════════
class ImageCache:
    def __init__(self, df: pd.DataFrame, img_size: int):
        self._cache: dict[str, np.ndarray] = {}
        paths = df["full_path"].unique().tolist()
        print(f"\n[CACHE] Pre-loading {len(paths):,} DICOMs into Kaggle RAM...")
        for p in tqdm(paths, desc="  Loading", unit="img"):
            self._cache[p] = load_dicom_as_rgb(p, img_size)
        mem_mb = sum(v.nbytes for v in self._cache.values()) / 1e6
        print(f"  Cache built — {mem_mb:.0f} MB used")

    def get(self, path: str) -> np.ndarray:
        return self._cache[path].copy()



In [4]:


# ══════════════════════════════════════════════════════════════
# CELL 7 — Patient-level split
# ══════════════════════════════════════════════════════════════
def patient_split(df: pd.DataFrame):
    print("\n[2] Patient-level split")
    subj_col = next((c for c in ["SubjectID", "PatientID", "subject_id"] if c in df.columns), None)
    
    if subj_col is None:
        print("  [warn] No SubjectID column — falling back to random row split")
        tr, tmp = train_test_split(df, test_size=Config.VAL_RATIO + Config.TEST_RATIO,
                                   stratify=df["label"], random_state=Config.SEED)
        va, te  = train_test_split(tmp, test_size=Config.TEST_RATIO / (Config.VAL_RATIO + Config.TEST_RATIO),
                                   stratify=tmp["label"], random_state=Config.SEED)
        return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)

    gss1 = GroupShuffleSplit(1, test_size=Config.TEST_RATIO, random_state=Config.SEED)
    tv_idx, te_idx = next(gss1.split(df, groups=df[subj_col]))
    df_tv, df_te   = df.iloc[tv_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)

    adj_val = Config.VAL_RATIO / (1 - Config.TEST_RATIO)
    gss2    = GroupShuffleSplit(1, test_size=adj_val, random_state=Config.SEED)
    tr_idx, va_idx = next(gss2.split(df_tv, groups=df_tv[subj_col]))
    df_tr, df_va   = df_tv.iloc[tr_idx].reset_index(drop=True), df_tv.iloc[va_idx].reset_index(drop=True)

    return df_tr, df_va, df_te


# ══════════════════════════════════════════════════════════════
# CELL 8 — Metadata encoder
# ══════════════════════════════════════════════════════════════
class MetadataEncoder:
    SITE = {"CFV": 0, "FV": 1, "PV": 2, "GS": 3}
    LIMB = {"R": 0, "L": 1}
    SEX  = {"F": 0, "M": 1}

    def __init__(self, df: pd.DataFrame):
        self.cont = [c for c in ["Age", "Height", "Weight", "Thigh Circumference"] if c in df.columns]
        self.scaler = StandardScaler()
        if self.cont:
            self.scaler.fit(df[self.cont].fillna(df[self.cont].mean()))

    @property
    def dim(self) -> int:
        return len(self.cont) + 4 + 2 + 2 + 1   

    def encode(self, row: pd.Series) -> np.ndarray:
        v = []
        if self.cont:
            vals = np.nan_to_num([float(row.get(c, 0)) for c in self.cont])
            v.extend(self.scaler.transform(vals.reshape(1, -1))[0])

        site_oh = [0.0] * 4
        site_oh[self.SITE.get(str(row.get("Anatomical Site", "CFV")).strip(), 0)] = 1.0
        v.extend(site_oh)

        limb_oh = [0.0, 0.0]
        limb_oh[self.LIMB.get(str(row.get("Limb", "R")).strip(), 0)] = 1.0
        v.extend(limb_oh)

        sex_oh = [0.0, 0.0]
        sex_oh[self.SEX.get(str(row.get("Sex", "F")).strip(), 0)] = 1.0
        v.extend(sex_oh)

        comp = float(row.get("Compressibility", 1.0))
        v.append(1.0 - comp / 2.0)   

        return np.array(v, dtype=np.float32)





In [5]:

# ══════════════════════════════════════════════════════════════
# CELL 9 — Dataset 
# ══════════════════════════════════════════════════════════════
class DVTDataset(Dataset):
    def __init__(self, df: pd.DataFrame, cache: ImageCache,
                 transform=None, meta_enc: MetadataEncoder = None):
        self.df        = df.reset_index(drop=True)
        self.cache     = cache
        self.transform = transform
        self.meta_enc  = meta_enc

    def get_sampler_weights(self) -> WeightedRandomSampler:
        labels = self.df["label"].values
        counts = np.bincount(labels, minlength=2).astype(float)
        w      = 1.0 / np.maximum(counts, 1)
        return WeightedRandomSampler(weights=torch.DoubleTensor(w[labels]), num_samples=len(labels), replacement=True)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx: int):
        row   = self.df.iloc[idx]
        label = int(row["label"])
        img   = self.cache.get(row["full_path"])    

        if self.transform: img = self.transform(image=img)["image"]
        meta = (torch.tensor(self.meta_enc.encode(row), dtype=torch.float32) if self.meta_enc else torch.zeros(1))
        return img, label, meta, row["full_path"]


# ══════════════════════════════════════════════════════════════
# CELL 10 — Transforms
# ══════════════════════════════════════════════════════════════
_NORM = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def get_transforms(split: str = "train") -> A.Compose:
    if split == "train":
        return A.Compose([
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
            A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.2), A.Rotate(limit=15, p=0.5),
            A.RandomBrightnessContrast(p=0.5),
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.35),
            A.GaussianBlur(blur_limit=(3, 5), p=0.2),
            A.CoarseDropout(max_holes=4, max_height=24, max_width=24, p=0.2),
            A.Normalize(**_NORM), ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
        A.Normalize(**_NORM), ToTensorV2(),
    ])

def build_loaders(train_df, val_df, test_df, cache, meta_enc):
    train_ds = DVTDataset(train_df, cache, get_transforms("train"), meta_enc)
    val_ds   = DVTDataset(val_df,   cache, get_transforms("val"),   meta_enc)
    test_ds  = DVTDataset(test_df,  cache, get_transforms("test"),  meta_enc)

    kw = dict(num_workers=Config.NUM_WORKERS, pin_memory=True) # pin_memory=True for GPU
    return (
        DataLoader(train_ds, Config.BATCH_SIZE, sampler=train_ds.get_sampler_weights(), **kw),
        DataLoader(val_ds,   Config.BATCH_SIZE, shuffle=False, **kw),
        DataLoader(test_ds,  Config.BATCH_SIZE, shuffle=False, **kw),
        test_ds,
    )


# ══════════════════════════════════════════════════════════════
# CELL 11 — Loss Functions
# ══════════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.75, smoothing=0.05):
        super().__init__()
        self.gamma, self.alpha, self.smoothing = gamma, alpha, smoothing

    def forward(self, logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        n = logits.size(1)
        with torch.no_grad():
            smooth = torch.full_like(logits, self.smoothing / (n - 1))
            smooth.scatter_(1, labels.unsqueeze(1), 1.0 - self.smoothing)

        log_p = F.log_softmax(logits, dim=1)
        pt    = (log_p.exp() * smooth).sum(dim=1)
        alpha_t = torch.where(labels == 1, torch.full_like(pt, self.alpha), torch.full_like(pt, 1.0 - self.alpha))
        return (-alpha_t * (1 - pt) ** self.gamma * (log_p * smooth).sum(dim=1)).mean()

class DVTLoss(nn.Module):
    def __init__(self, pos_weight: float):
        super().__init__()
        self.focal     = FocalLoss(Config.FOCAL_GAMMA, Config.FOCAL_ALPHA, Config.LABEL_SMOOTHING)
        self.conf_loss = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))

    def forward(self, out: dict, labels: torch.Tensor):
        cls  = self.focal(out["cls_logits"], labels)
        flag = (labels > 0).float()
        self.conf_loss.pos_weight = self.conf_loss.pos_weight.to(labels.device)
        conf = self.conf_loss(out["conf_logit"].squeeze(1), flag)
        return cls + 0.3 * conf


# ══════════════════════════════════════════════════════════════
# CELL 12 — Model 
# ══════════════════════════════════════════════════════════════
class DVTSwin(nn.Module):
    def __init__(self, num_classes: int = 2, meta_dim: int = 0):
        super().__init__()
        self.use_meta = Config.USE_METADATA and meta_dim > 0

        self.backbone = timm.create_model(Config.MODEL_NAME, pretrained=Config.PRETRAINED, num_classes=0, global_pool="")
        C = self.backbone.num_features   

        self.img_proj = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.LayerNorm(C), nn.Dropout(p=0.4), nn.Linear(C, 512), nn.GELU(),
        )

        meta_out = 0
        if self.use_meta:
            meta_out = 128
            self.meta_proj = nn.Sequential(
                nn.Linear(meta_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(p=0.2), nn.Linear(128, 128), nn.GELU(),
            )

        self.classifier = nn.Sequential(nn.Dropout(p=0.2), nn.Linear(512 + meta_out, 256), nn.GELU(), nn.Linear(256, num_classes))
        self.conf_head  = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(C, 64), nn.GELU(), nn.Linear(64, 1))
        self.hmap_head  = nn.Sequential(nn.Conv2d(C, 128, 1), nn.BatchNorm2d(128), nn.GELU(), nn.Conv2d(128, 1, 1), nn.Sigmoid())

    def _feats(self, x: torch.Tensor) -> torch.Tensor:
        f = self.backbone.forward_features(x)
        if f.dim() == 4: return f.permute(0, 3, 1, 2).contiguous()
        B, N, C = f.shape
        H = W = int(math.sqrt(N))
        return f.reshape(B, H, W, C).permute(0, 3, 1, 2).contiguous()

    def forward(self, x: torch.Tensor, meta: torch.Tensor = None) -> dict:
        feats = self._feats(x)                                      
        img   = self.img_proj(feats)                                
        fused = (torch.cat([img, self.meta_proj(meta)], dim=1) if self.use_meta and meta is not None else img)
        hmap = F.interpolate(self.hmap_head(feats), size=(Config.IMG_SIZE, Config.IMG_SIZE), mode="bilinear", align_corners=False)
        
        return {"cls_logits": self.classifier(fused), "conf_logit": self.conf_head(feats), "heatmap": hmap}

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
        print("  Backbone FROZEN")

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True
        print("  Backbone UNFROZEN")


In [6]:

# ══════════════════════════════════════════════════════════════
# CELL 13 — Trainer (AMP + DataParallel Aware)
# ══════════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, train_loader, val_loader, output_dir: str):
        self.model        = model.to(Config.DEVICE)
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.output_dir   = output_dir
        self.best_val_auc = 0.0
        self.history      = {k: [] for k in ["train_loss", "val_loss", "train_acc", "val_acc", "val_auc"]}
        
        self.scaler = GradScaler() # Restored for GPU speedup
        os.makedirs(output_dir, exist_ok=True)
        self.criterion = DVTLoss(pos_weight=Config.DVT_POS_WEIGHT).to(Config.DEVICE)

        self._build_optimizer_and_scheduler(
            params=filter(lambda p: p.requires_grad, model.parameters()),
            n_steps=len(train_loader) * Config.NUM_EPOCHS,
        )

    def _get_base_model(self):
        # Unwraps DataParallel model to access custom methods/attributes
        return self.model.module if isinstance(self.model, nn.DataParallel) else self.model

    def _build_optimizer_and_scheduler(self, params, n_steps: int):
        self.optimizer = torch.optim.AdamW(list(params), lr=Config.LR_MAX / 10, weight_decay=Config.WEIGHT_DECAY)
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer, max_lr=Config.LR_MAX, total_steps=n_steps,
            pct_start=0.15, anneal_strategy="cos", div_factor=10, final_div_factor=100,
        )

    def _run_epoch(self, loader, train: bool):
        self.model.train(train)
        total_loss = correct = total = 0
        all_probs, all_labels = [], []

        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for imgs, labels, meta, _ in loader:
                imgs, labels, meta = imgs.to(Config.DEVICE), labels.to(Config.DEVICE), meta.to(Config.DEVICE)

                if train: self.optimizer.zero_grad()

                # AMP Autocast context for faster GPU computation
                with autocast():
                    out  = self.model(imgs, meta)
                    loss = self.criterion(out, labels)

                if train:
                    self.scaler.scale(loss).backward()
                    self.scaler.unscale_(self.optimizer)
                    nn.utils.clip_grad_norm_(self.model.parameters(), Config.GRAD_CLIP)
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.scheduler.step()

                probs = F.softmax(out["cls_logits"], dim=1)
                preds = probs.argmax(1)
                total_loss += loss.item()
                correct    += (preds == labels).sum().item()
                total      += labels.size(0)
                all_probs.extend(probs.detach().cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        all_probs, all_labels = np.array(all_probs), np.array(all_labels)
        roc_auc = roc_auc_score(all_labels, all_probs[:, 1]) if len(np.unique(all_labels)) > 1 else 0.5
        return total_loss / len(loader), correct / total, roc_auc

    def train(self):
        print(f"\n[5] Training on {Config.DEVICE} | Epochs={Config.NUM_EPOCHS}")
        print("=" * 65)

        self._get_base_model().freeze_backbone()

        for epoch in range(1, Config.NUM_EPOCHS + 1):
            if epoch == Config.FREEZE_EPOCHS + 1:
                self._get_base_model().unfreeze_backbone()
                self._build_optimizer_and_scheduler(params=self.model.parameters(), n_steps=len(self.train_loader) * (Config.NUM_EPOCHS - Config.FREEZE_EPOCHS))

            t_loss, t_acc, _      = self._run_epoch(self.train_loader, train=True)
            v_loss, v_acc, v_auc  = self._run_epoch(self.val_loader,   train=False)

            for k, val in [("train_loss", t_loss), ("val_loss", v_loss), ("train_acc",  t_acc),  ("val_acc",  v_acc), ("val_auc",    v_auc)]:
                self.history[k].append(val)

            marker = ""
            if v_auc > self.best_val_auc:
                self.best_val_auc = v_auc
                # Save base model to avoid 'module.' prefix issues
                torch.save({"epoch": epoch, "state_dict": self._get_base_model().state_dict(), "val_auc": v_auc}, f"{self.output_dir}/best_model.pth")
                marker = "  ✓ saved"

            phase = "FREEZE" if epoch <= Config.FREEZE_EPOCHS else "TUNE  "
            print(f"  Epoch [{epoch:02d}/{Config.NUM_EPOCHS}] [{phase}] loss={t_loss:.4f}/{v_loss:.4f} acc={t_acc:.3f}/{v_acc:.3f} AUC={v_auc:.4f} {marker}")

        return self.history


# ══════════════════════════════════════════════════════════════
# CELL 14, 15, 16 — Metrics & Visualization (Condensed)
# ══════════════════════════════════════════════════════════════
CLASS_NAMES = ["Normal", "DVT"]

@torch.no_grad()
def clinical_evaluation(model, test_loader, output_dir: str, threshold: float = None):
    print("\n[6] Clinical evaluation on test set")
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    for imgs, labels, meta, _ in tqdm(test_loader, desc="  testing"):
        out   = model(imgs.to(Config.DEVICE), meta.to(Config.DEVICE))
        probs = F.softmax(out["cls_logits"], dim=1)
        all_preds.extend(probs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

    all_probs, all_labels = np.array(all_probs), np.array(all_labels)
    dvt_score = all_probs[:, 1]
    fpr, tpr, thresholds = roc_curve(all_labels, dvt_score)
    roc_auc = auc(fpr, tpr)

    best_idx = int(np.argmax(tpr - fpr))
    opt_thr  = float(thresholds[best_idx])
    thr = threshold if threshold is not None else opt_thr

    final_preds = (dvt_score >= thr).astype(int)
    cm = confusion_matrix(all_labels, final_preds)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)

    sens, spec = tp / max(tp + fn, 1), tn / max(tn + fp, 1)
    ppv, npv = tp / max(tp + fp, 1), tn / max(tn + fn, 1)

    print(f"\n  ROC-AUC: {roc_auc:.4f} | Sens: {sens:.3f} | Spec: {spec:.3f} | PPV: {ppv:.3f} | NPV: {npv:.3f}")
    return {"roc_auc": roc_auc, "sensitivity": sens, "specificity": spec, "ppv": ppv, "npv": npv, "threshold": thr}

@torch.no_grad()
def visualize_heatmaps(model, test_ds, n: int = 8, output_dir: str = ".", threshold: float = 0.5):
    model.eval()
    indices = np.random.choice(len(test_ds), min(n, len(test_ds)), replace=False)
    _mean, _std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

    for k, idx in enumerate(indices):
        img_t, label, meta, path = test_ds[idx]
        out   = model(img_t.unsqueeze(0).to(Config.DEVICE), meta.unsqueeze(0).to(Config.DEVICE))
        probs = F.softmax(out["cls_logits"], dim=1)[0].cpu().numpy()
        hmap  = out["heatmap"][0, 0].cpu().numpy()

        img_np   = np.clip((img_t.permute(1, 2, 0).numpy() * _std + _mean) * 255, 0, 255).astype(np.uint8)
        hmap_jet = cv2.cvtColor(cv2.applyColorMap((hmap * 255).astype(np.uint8), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
        overlay  = cv2.addWeighted(img_np, 0.6, hmap_jet, 0.4, 0)

        fig, axes = plt.subplots(1, 3, figsize=(13, 4))
        axes[0].imshow(img_np); axes[0].axis("off")
        axes[1].imshow(hmap, cmap="hot"); axes[1].axis("off")
        axes[2].imshow(overlay); axes[2].axis("off")
        
        plt.savefig(f"{output_dir}/case_{k+1:02d}.png", dpi=150, bbox_inches="tight")
        plt.close()


In [7]:

# ══════════════════════════════════════════════════════════════
# CELL 17 — MAIN 
# ══════════════════════════════════════════════════════════════
def main():
    torch.manual_seed(Config.SEED)
    np.random.seed(Config.SEED)
    os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

    df = build_master_dataframe(Config.DICOM_DIR, Config.CSV_PATH)
    validate_images(df, n_check=30)
    cache = ImageCache(df, Config.IMG_SIZE)
    train_df, val_df, test_df = patient_split(df)
    
    meta_enc = MetadataEncoder(train_df)
    train_loader, val_loader, test_loader, test_ds = build_loaders(train_df, val_df, test_df, cache, meta_enc)

    # ── Initialize and Wrap Model in DataParallel ────────────
    model = DVTSwin(num_classes=2, meta_dim=meta_enc.dim if Config.USE_METADATA else 0)
    
    if torch.cuda.device_count() > 1:
        print(f"\n  Wrapping model in nn.DataParallel across {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
        
    model = model.to(Config.DEVICE)

    # ── Train ──────────────────────────────────────────────
    trainer = Trainer(model, train_loader, val_loader, Config.OUTPUT_DIR)
    trainer.train()

    # ── Load BEST checkpoint ───────────────────────────────
    ckpt = torch.load(f"{Config.OUTPUT_DIR}/best_model.pth", map_location=Config.DEVICE)
    base_model = model.module if isinstance(model, nn.DataParallel) else model
    base_model.load_state_dict(ckpt["state_dict"])
    
    print(f"\n  Loaded best model — epoch {ckpt['epoch']} val_AUC={ckpt['val_auc']:.4f}")

    # ── Eval & Visuals ─────────────────────────────────────
    results = clinical_evaluation(model, test_loader, Config.OUTPUT_DIR, threshold=Config.DECISION_THRESHOLD)
    visualize_heatmaps(model, test_ds, n=8, output_dir=Config.OUTPUT_DIR, threshold=results["threshold"])
    print(f"\n  Pipeline Complete. Output artifacts located in {Config.OUTPUT_DIR}")

if __name__ == "__main__":
    main()


[1] Building master dataframe (CSV ↔ DICOM join)
  CSV rows       : 2,341
  DICOM on disk  : 2,341
  Normal: 2179  |  DVT:  162  |  Ratio=13.5:1

[IMG CHECK] Sampling 30 DICOMs...


error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:3845: error: (-215:Assertion failed) !dsize.empty() in function 'resize'
